#   Landing to Bronze


## 1. Parâmetros do Ambiente - Organização do Ambiente

In [0]:
#PADRÃO: <catalog>.<camada>.<tabela>
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

##2. Catalog, Schemas e Volumes

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Criação de catalog e schemas finalizada")

## 3. Upload dos Arquivos

In [0]:
#Mapeamento dos caminhos no Volume
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"



#Leitura pura (sem transformações de negócio)
#Aqui optei por não usar inferSchema=True. Tomei essa decisão porque o enunciado descreve os dados como "intencionalmente sujos e fragmentados".

#Como o inferSchema tenta identificar automaticamente o tipo de cada coluna, fiquei com receio de o Spark definir um tipo incorreto e, na hora da leitura, os valores que não correspondessem a esse tipo fossem convertidos para null silenciosamente. Isso poderia acabar alterando ou perdendo dados já na ingestão.

#Como a camada Bronze deve manter os dados sem alterações, achei mais seguro deixar tudo como string nessa etapa. Assim, nenhum dado é perdido por causa de uma conversão automática e os tipos e tratamentos podem ser definidos depois, na camada Silver.
mapeamento = {
    path_movies_info: f"{bronze_schema}.tb_movies_info",
    path_movies_metrics: f"{bronze_schema}.tb_movies_metrics",
    path_movies_reviews: f"{bronze_schema}.tb_movies_reviews",
    path_movies_financials: f"{bronze_schema}.tb_movies_financials",
    path_credits_and_tags: f"{bronze_schema}.tb_credits_and_tags"
}
dicionario_dfs = {}
for path, tabela in mapeamento.items():
    dicionario_dfs[tabela] = spark.read.csv(path, header=True)

print(f"Conteudo dicionário: {dicionario_dfs}")

## 4. Validação da landing 

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv",
    "movies_financials_IMDB_TMDB.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")


# Verificações adicionais da landing zone
# Antes de seguir para a gravação,  tomei a decisão de conferir linhas, colunas e nulos no id de cada tabela, 
# para garantir que a leitura ocorreu sem falhas silenciosas e facilitar uma visão geral dessa análise.
for tabela, df in dicionario_dfs.items():
    print(f"\n {tabela.upper()}:\n LINHAS: {df.count()} linhas. | COLUNAS: {len(df.columns)} colunas. | ID's Nulos: {df.filter(df.id.isNull()).count()} linhas nulas. ")
    print("________________________________________________________________________________")
    
        

## Observação sobre a validação da landing zone

Ao analisar a tabela de validação, percebi que a contagens de linha divergem entre as tabelas que deveriam ter um registro por filme (tb_movies_info: 106.930, tb_movies_metrics: 107.364, tb_movies_financials: 106.165, tb_credits_and_tags: 106.320).
Acredito que isso é um indício de ids duplicados ou filmes ausentes e isso parece coerente com a natureza suja dos dados. Como a Bronze deve preservar os dados sem alterações, esse tratamento fica para a Silver, na etapa de deduplicação por filme. A tb_movies_reviews (32.412 linhas) tem granularidade diferente (por avaliação, não por filme), então não considerei nessa comparação.

## 5. Ingestão Bronze
Leitura pura da Landing e gravação em Delta na Bronze, sem nenhuma regra de negócio.
É adicionado apenas o ingestion_datetime para registrar o momento exato em que o dado é inserido na camada Bronze.

In [0]:
from pyspark.sql.functions import current_timestamp

#Gravação com adição do timestamp no momento da escrita.
#Decidi gravar os dados com um loop para evitar a repetição de código.
for tabela , df in dicionario_dfs.items():
    df \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(tabela)

display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))

## 6. Ingestão de API: 
Fase da extração da cotação do dólar via API do Banco Central. 
Percebi que a série de cotações retornada pela API não cobre finais de semana e feriados. Essa lacuna era esperada e será tratada na camada Silver com preenchimento por forward-fill, conforme especificado no enunciado.

In [0]:
from datetime import datetime, timedelta

#Seguindo a orientação, estou calculando a data para consultar os últimos 7 dias corridos a partir da data de execução.
data_fim = datetime.now()
data_inicio = data_fim - timedelta(days=7)

data_fim_formatada = data_fim.strftime("%m-%d-%Y")
data_inicio_formatada = data_inicio.strftime("%m-%d-%Y")

# Utilizei os parâmetros widget como solcitado e fiz de forma que caso nada seja inserido manualmente, a data_inicio e data_fim calculadas 
#automaticamente já são iseridas.
dbutils.widgets.text("widget_data_inicio", data_inicio_formatada , "DATA INÍCIO (MM-DD-AAAA):")
valor_inicio = dbutils.widgets.get("widget_data_inicio")

dbutils.widgets.text("widget_data_fim", data_fim_formatada , "DATA FIM (MM-DD-AAAA):")
valor_fim = dbutils.widgets.get("widget_data_fim")

#Decidi colocar esses blocos try/except para caso a data nos widget não seja no formato especificado pela documentação da atividade (MM-DD-AAAA).
try:
    datetime.strptime(valor_inicio, "%m-%d-%Y")
except ValueError:
    raise ValueError("A data de início está com formato inválido. Use MM-DD-AAAA.")

try:
    datetime.strptime(valor_fim, "%m-%d-%Y")
except ValueError:
    raise ValueError("A data de fim está com formato inválido. Use MM-DD-AAAA.")




In [0]:
import requests
from pyspark.sql.functions import current_timestamp

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{valor_inicio}'&@dataFinalCotacao='{valor_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

resposta = requests.get(url)

#Salvando em uma tabela a cotação do dolar e verificando o status code.
if resposta.status_code == 200:
    cotacoes = resposta.json()
    cotacoes = cotacoes['value']
    cotacoes = spark.createDataFrame(cotacoes)
    cotacoes.withColumn("ingestion_datetime", current_timestamp()).write.format("delta").mode("append").saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
else:
    raise Exception(f"Erro ao fazer a requisição: {resposta.status_code}")

display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").limit(5))